# Extraction des projections climatiques Explore2 (TRACC) — 5 communes pilotes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarcoDamz/bati-scolaire-climat/blob/claude/enrich-demo-instances-oncka4/notebooks/01_extraction_explore2.ipynb)

Organisation du projet : **GitHub** contient le code, **Google Drive** les données, **Colab** exécute.

Ce notebook lit sur le serveur public du CNRM (Météo-France, sans compte) les projections journalières
Explore2-2022 corrigées ADAMONT (17 simulations, historique + RCP8.5), et n'en garde que les mailles de 8 km
qui contiennent les bâtiments des 5 pilotes. Résultat écrit dans
`Mon Drive/SaaS adaptation climatique/demo/donnees/explore2/`.

**Durée : plusieurs heures.** Si Colab se déconnecte, relancez simplement toutes les cellules :
les fichiers déjà traités sont conservés sur le Drive et sautés.

## 1. Paramètres et Google Drive

In [ ]:
DOSSIER_DRIVE = "/content/drive/MyDrive/SaaS adaptation climatique/demo/donnees/explore2"
DEPOT = "https://github.com/MarcoDamz/bati-scolaire-climat.git"
BRANCHE = "claude/enrich-demo-instances-oncka4"   # passer à "main" une fois la branche fusionnée

from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DOSSIER_DRIVE, exist_ok=True)
print("Sortie :", DOSSIER_DRIVE)

## 2. Installation et code depuis GitHub

In [ ]:
!pip install -q xarray h5netcdf h5py netCDF4 fsspec aiohttp
import os, sys, subprocess
CODE = "/content/bati-scolaire-climat"
if os.path.isdir(CODE):
    subprocess.run(["git", "-C", CODE, "fetch", "-q", "origin", BRANCHE], check=True)
    subprocess.run(["git", "-C", CODE, "checkout", "-q", "-B", BRANCHE, f"origin/{BRANCHE}"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCHE, DEPOT, CODE], check=True)
sys.path.insert(0, f"{CODE}/phase1")
import importlib, telecharger_explore2 as t
importlib.reload(t)
print("Code à jour :", subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

## 3. Le serveur du CNRM répond-il depuis Colab ?

Il filtre certains hébergeurs cloud. S'il ne répond pas, lancez le même script depuis votre ordinateur
(`python3 phase1/telecharger_explore2.py --dossier "<dossier Google Drive synchronisé>/demo/donnees/explore2"`).

In [ ]:
acces = t.tester_acces()
assert acces, "Serveur CNRM inaccessible depuis Colab : voir l'alternative ci-dessus." 

## 4. Ce qui sera extrait (quelques minutes)

In [ ]:
bilan = t.executer(DOSSIER_DRIVE, lister=True)
bilan.groupby("variable")["chaine"].nunique()

## 5. Extraction (plusieurs heures, reprise automatique)

Attendu : 17 chaînes pour chaque variable. Si ce n'est pas le cas, envoyez `explore2_fichiers_retenus.csv`
(dans le dossier Drive) avant de lancer cette cellule.

In [ ]:
donnees = t.executer(DOSSIER_DRIVE)
donnees.groupby(["variable", "scenario"], observed=True)["chaine"].nunique()